# Design and Algorithms of `BandProjected_ExactDiagonalization`

This notebook documents the mathematical formulation, data structures, and performance-critical pure-NumPy/SciPy algorithms powering `bandprojected_exactdiagonalization`. All kernels are strictly NumPy/SciPy (no `numba`), supporting arbitrary multi-band momentum-space projected Hamiltonians (band, valley, sublattice, spin), diagonal and sheared/tilted 2D Bravais tori, sparse CSR vs zero-loop BLAS-3 matrix-free Krylov eigensolvers, exact Fock-space particle-hole duality, and both 1D Resta–Thouless and 2D FHS topological invariants.

## 1. Bravais Lattices and Smith Normal Form (SNF) Sheared Tori

Given primitive Bravais row vectors $\mathbf{A} = \begin{pmatrix}\mathbf{a}_1 \\ \mathbf{a}_2\end{pmatrix}$ and an arbitrary positive-determinant integer supercell matrix $\mathbf{N} \in \mathbb{Z}^{2\times 2}$ defining sample torus periods $\mathbf{L} = \mathbf{N}\mathbf{A}$, the reciprocal lattices satisfy $\mathbf{A}\mathbf{B}^T = 2\pi\mathbf{I}_2$ and $\mathbf{L}\mathbf{B}_{\mathrm{SC}}^T = 2\pi\mathbf{I}_2$, with $\mathbf{B}_{\mathrm{SC}} = \mathbf{N}^{-T}\mathbf{B}$.

To index the $N_{\mathrm{uc}} = |\det\mathbf{N}|$ finite-torus crystal momenta canonically without hardcoding rectangular axes, `smith_normal_form_2x2` computes unimodular integer matrices $\mathbf{U}, \mathbf{V} \in \mathrm{GL}_2(\mathbb{Z})$ such that

\begin{equation}
\mathbf{U}\,\mathbf{N}\,\mathbf{V} = \mathbf{D} = \operatorname{diag}(d_1, d_2), \qquad N_{\mathrm{uc}} = d_1 d_2.
\end{equation}

Canonical SNF integer momentum labels $\mathbf{s} = (s_1, s_2) \in [0, d_1)\times[0, d_2)$ map to primitive reciprocal fractional coordinates via $\mathbf{k}_{\mathrm{frac}} = (\mathbf{s}\mathbf{D}^{-1}\mathbf{V}^T + \tfrac{\boldsymbol{\phi}}{2\pi}\mathbf{N}^{-T}) \bmod 1$, preserving exact $(k_1, k_2)$ indexing whenever $\mathbf{N} = \operatorname{diag}(N_1, N_2)$ ($\mathbf{U}=\mathbf{V}=\mathbf{I}_2$). Every sample transfer $\mathbf{q} \in \mathbb{Z}^2$ shifts SNF sectors by $\mathbf{q}_{\mathrm{SNF}} = \mathbf{q}\mathbf{U}^T \pmod{(d_1, d_2)}$ while returning exact integer Umklapp reciprocal windings $\mathbf{w}_{\mathbf{k}}(\mathbf{q}) \in \mathbb{Z}^2$.

In [1]:
import numpy as np
from bandprojected_exactdiagonalization import BravaisLattice, TorusGeometry, smith_normal_form_2x2

lat = BravaisLattice.triangular(1.0, 60.0)
# Paper cTDBG tilted 24-cell torus [[5, 1], [1, 5]] vs [[1, 5], [0, 24]]
for N in [np.array([[5, 1], [1, 5]]), np.array([[1, 5], [0, 24]]), np.array([4, 6])]:
    geom = TorusGeometry(lat, N)
    tgt, w = geom.shift((1, 2))
    print(f"N={N.tolist()} -> SNF shape={geom.shape}, size={geom.size}, is_diag={geom.is_diagonal}, max|w|={np.abs(w).max()}")

N=[[5, 1], [1, 5]] -> SNF shape=(1, 24), size=24, is_diag=False, max|w|=1
N=[[1, 5], [0, 24]] -> SNF shape=(1, 24), size=24, is_diag=False, max|w|=1
N=[4, 6] -> SNF shape=(4, 6), size=24, is_diag=True, max|w|=1


## 2. Bloch Manifolds, Zero-Padded 2D FFT Form Factors, and Exact PH Duality

In our Fourier convention, physical Bloch states carry wavevector $-\mathbf{k}$ and read $\psi_{\mathbf{k},a}(\mathbf{r}) = e^{-i\mathbf{k}\cdot\mathbf{r}} \sum_{\mathbf{G},c} u_{\mathbf{k},(\mathbf{G},c),a}\, e^{i\mathbf{G}\cdot\mathbf{r}} / \sqrt{A}$, so the projected density operator $\hat{\rho}_{\mathbf{q}} = \hat{P}e^{-i\mathbf{q}\cdot\mathbf{r}}\hat{P}$ scatters $\mathbf{k} \to [\mathbf{k}+\mathbf{q}]$ with multi-band vertex matrix

\begin{equation}
\Lambda_{a,b}(\mathbf{k}, \mathbf{q}) = \sum_{\mathbf{G},c} u^{*}_{[\mathbf{k}+\mathbf{q}],(\mathbf{G}-\mathbf{w}_{\mathbf{k}}(\mathbf{q}),c),a}\, u_{\mathbf{k},(\mathbf{G},c),b}.
\end{equation}

### Batch Evaluation via Zero-Padded Linear FFT Correlation

Evaluating Eq. (2) transfer-by-transfer scales poorly for thousands of $\mathbf{q}$ vectors in the interaction disk $q\ell \le q_{\max}\ell$. Instead, `BlochBand.form_factors` packs $u_{\mathbf{k},(\mathbf{G},c),a}$ onto a zero-padded spatial grid of size $\ge 2W_1-1 \times 2W_2-1$ (eliminating circular wraparound aliasing identically), transforms once along reciprocal indices, and evaluates each distinct SNF residue class via a single `np.fft.ifft2`, cutting complexity from $O(N_{\mathbf{q}} N_{\mathrm{pw}})$ to $O(N_{\mathrm{uc}} N_{\mathrm{pad}} \log N_{\mathrm{pad}})$.

### Antiunitary Particle-Hole Transformation

For any filled-band self-energy $\Sigma_{\mathrm{full}}(\mathbf{k})$ constructed from closed Hartree ($\mathbf{q}_{\mathrm{SNF}} \equiv \mathbf{0}$) and Fock exchange loops, `BlochBand.particle_hole_transform` maps $N_{\mathrm{orb}} - N_h$ electrons to $N_h$ partners under the **same two-body vertices** and one-body matrix $h_{\mathrm{PH}}(\mathbf{k}) = -h_e(\mathbf{k}) - \Sigma_{\mathrm{full}}(\mathbf{k})$, yielding the exact operator identity $E(N_{\mathrm{orb}} - N_h) = E_{\mathrm{full}} + E_{\mathrm{PH}}(N_h)$ with $E_{\mathrm{full}} = \operatorname{Tr}(h_e + \tfrac{1}{2}\Sigma_{\mathrm{full}})$.

In [2]:
from bandprojected_exactdiagonalization.models import IntervalleyTBG

ic_band = IntervalleyTBG(cutoff=3.5).band((4, 6))
qs = ic_band.geometry.transfer_disk(4.0)
ff_batch = ic_band.form_factors(qs)
ff_direct = np.stack([ic_band.form_factor(q) for q in qs], axis=0)
print("Max FFT vs direct form-factor error:", float(np.max(np.abs(ff_batch - ff_direct))))

Max FFT vs direct form-factor error: 1.4433741427748226e-15


## 3. Bitwise Fock Space, Meet-in-the-Middle Enumeration, and `SectorLookup`

Every $N_e$-fermion Slater determinant over $N_{\mathrm{orb}} \le 63$ spin-orbitals is stored as a sorted ascending `np.uint64` bitmask $m = \sum_{j=0}^{N_{\mathrm{orb}}-1} n_j 2^j$.

1. **Meet-in-the-Middle Sector Catalog (`MomentumSectorCatalog`)**: Rather than scanning all $\binom{N_{\mathrm{orb}}}{N_e}$ configurations globally (which exceeds $10^6$ even for modest $p4mm$ or $4\times 6$ problems), orbitals are split into left ($n_L = \lfloor N_{\mathrm{orb}}/2 \rfloor$) and right ($n_R = N_{\mathrm{orb}} - n_L$) halves, bucketed once by $(n, \mathbf{k} \bmod \mathbf{D})$, and joined in exact momentum pairs $(n_L, \mathbf{K}_L) + (N_e - n_L, \mathbf{K} - \mathbf{K}_L)$.
2. **Branch-Free Jordan–Wigner Signs (`parity_sign_u64`)**: Precomputing `between[i, j]` masks with bits strictly between $i$ and $j$, creation/annihilation fermion signs reduce to a single bitwise AND + hardware `np.bitwise_count` parity with zero branching.
3. **Hybrid `SectorLookup`**: Uses an $O(1)$ prefix array when bucket occupancy $\le 1$, and C-level GIL-releasing `np.searchsorted` otherwise.

In [3]:
import math
from bandprojected_exactdiagonalization import MomentumSectorCatalog, BandProjectedED
from bandprojected_exactdiagonalization.models import TorusLLL, haldane_v1_hamiltonian

lll_band, lll_ham = haldane_v1_hamiltonian(shape=(3, 4), qell_cutoff=7.5, include_q0=True)
cat = MomentumSectorCatalog(12, (3, 4), lll_band.geometry.indices, max_particles=4)
b0 = cat.basis(4, (0, 0))
print(f"3x4 LLL N_e=4: full C(12,4)={math.comb(12, 4)}, sector (0,0) dim={len(b0)}")
print("Trace defect on 12x12 LLL:", float(TorusLLL(chirality=1).band((12, 12)).quantum_geometry()["trace_defect"].max()))

3x4 LLL N_e=4: full C(12,4)=495, sector (0,0) dim=42
Trace defect on 12x12 LLL: 0.00021154658722188913


## 4. Momentum-Blocked Sparse CSR & Zero-Loop BLAS-3 Matrix-Free Solvers

Any projected two-body interaction is canonically antisymmetrized over ordered pairs $\alpha = (i_1 < i_2)$ grouped by conserved total pair SNF momentum $\mathbf{P} = \mathbf{k}_{i_1} + \mathbf{k}_{i_2} \pmod{\mathbf{D}}$ into Hermitian blocks $\mathbf{W}^{(\mathbf{P})}$:

\begin{equation}
\hat{H}_2 = \sum_{\mathbf{P}} \sum_{\alpha,\beta \in \mathrm{pairs}(\mathbf{P})} W^{(\mathbf{P})}_{\alpha,\beta}\, c^\dagger_{i_1(\alpha)} c^\dagger_{i_2(\alpha)} c_{i_2(\beta)} c_{i_1(\beta)}.
\end{equation}

- **Sparse CSR (`mode="matrix"`)**: Assembles **strictly half** ($\alpha > \beta$ and $i_{\mathrm{to}} < i_{\mathrm{from}}$) of the off-diagonal terms and constructs $\hat{H}_{\mathrm{CSR}} = \hat{H}_{\mathrm{half}} + \hat{H}_{\mathrm{half}}^\dagger + \hat{H}_{\mathrm{diag}}$, halving bitmask lookups and temporary COO memory while guaranteeing exact Hermiticity.
- **Zero-Loop BLAS-3 `ZGEMM + bincount` Matrix-Free (`mode="matrixfree"`)**: For each pair block $\mathbf{P}$ and chunk of the $(N_e-2, \mathbf{K}-\mathbf{P})$ intermediate spectator masks $r$, six pure-NumPy kernels execute without Python inner loops:
  1. Disjoint spectator filter `(pos, col) = np.nonzero((rc[:, None] & pmasks[None, :]) == 0)`
  2. Binary search target indices `idx = np.searchsorted(basis.masks, r[pos] | pmasks[col])`
  3. Branch-free fermion signs `sgn = parity_sign_u64(r[pos] & btw[col])`
  4. Gather `X[pos, col] = sgn * vec[idx]`
  5. Single BLAS-3 multiply `Y = X @ W_off_P.T`
  6. Vectorized `np.bincount` real/imag accumulation back into `out` (replaces slow `np.add.at`).
- **ARPACK-Safe `sector_workers`**: Because Fortran ARPACK uses non-reentrant static workspace, `BandProjectedED.scan_sectors(..., sector_workers=W)` parallelizes independent momentum sectors across `ProcessPoolExecutor` workers initialized once with `num_threads=1` and `OPENBLAS_NUM_THREADS=1`.

## 5. Topological & Many-Body Diagnostics: 1D Charge Pump, PES, and Benchmarks

In [4]:
import json
from pathlib import Path

bench = json.loads(Path("benchmark/benchmark_results.json").read_text())
print("Size Scaling Summary:")
for row in bench["size_scaling"]:
    lbl = row["label"].replace("\n", " ")
    m, mf = row["matrix"], row["matrixfree"]
    print(f"  {lbl:25s} | CSR: {m['solve_seconds']:6.3f}s ({m['peak_memory_mb']:7.1f} MB) | MF: {mf['solve_seconds']:6.3f}s ({mf['peak_memory_mb']:6.1f} MB)")

print("\nMatrix-Free Thread Scaling (p4mm D=61,333):")
for row in bench["thread_scaling"]:
    print(f"  nt={row['threads']:2d} -> {row['matvec_seconds']*1e3:6.2f} ms/matvec")

print("\ncTDBG 1D Resta-Thouless Charge Pump Chern:", bench["ctdbg_pumps"])

Size Scaling Summary:
  tMoTe2 3x5 (D=200)        | CSR:  0.067s (    0.5 MB) | MF:  0.452s (   0.4 MB)
  tMoTe2 3x6 (D=1,031)      | CSR:  0.149s (    4.1 MB) | MF:  0.921s (   2.3 MB)
  tMoTe2 4x6 (D=30,667)     | CSR:  5.050s (  299.7 MB) | MF:  8.901s (  73.3 MB)
  p4mm 3x4 (D=61,333)       | CSR: 13.671s ( 1220.4 MB) | MF: 10.552s ( 255.7 MB)

Matrix-Free Thread Scaling (p4mm D=61,333):
  nt= 1 -> 219.46 ms/matvec
  nt= 2 -> 125.31 ms/matvec
  nt= 4 ->  66.90 ms/matvec
  nt= 8 ->  46.04 ms/matvec
  nt=12 ->  35.14 ms/matvec

cTDBG 1D Resta-Thouless Charge Pump Chern: {'beta0p1_chern': 1, 'beta1p0_chern': 2}


### Verified 1D Resta–Thouless Charge Pump vs 2D FHS Chern Cross-Check

In [5]:
from bandprojected_exactdiagonalization import thouless_charge_pump_1d, many_body_chern_number_and_pump

# Fast exact LLL Haldane V1 check along BOTH axes
p_y = thouless_charge_pump_1d(
    lambda tw: haldane_v1_hamiltonian(shape=(2, 3), qell_cutoff=7.5, twists=tw, include_q0=True),
    n_particles=2, n_steps=8, flux_axis=1, manifold_size=3, nev=3,
)
p_x = thouless_charge_pump_1d(
    lambda tw: haldane_v1_hamiltonian(shape=(3, 4), qell_cutoff=7.5, twists=tw, include_q0=True),
    n_particles=4, n_steps=8, flux_axis=0, manifold_size=3, nev=3,
)
print(f"LLL (2,3) flux_axis=1: C={p_y['integer_chern']}, sigma_xy={p_y['averaged_hall_conductivity']:.4f}, resolved={p_y['resolved']}")
print(f"LLL (3,4) flux_axis=0: C={p_x['integer_chern']}, sigma_xy={p_x['averaged_hall_conductivity']:.4f}, resolved={p_x['resolved']}")

LLL (2,3) flux_axis=1: C=1, sigma_xy=0.3333, resolved=True
LLL (3,4) flux_axis=0: C=1, sigma_xy=0.3333, resolved=True
